In [ ]:
import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit, ols

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# загружен датасеты в виде словаря
path_to_datasets = '../datasets/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
		'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
		'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

## Качество подгонки

In [ ]:
df_names = ['loanapp', 'loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'TableF5-1', 'SwissLabour', 'SwissLabour']

endog_var = ['approve', 'approve', 'approve', 'LFP', 'LFP', 'LFP', 'participation', 'participation']

exog_vars =['appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self', 
			'mortno+unem+dep+male+married+yjob+self',
			'appinc+I(appinc**2)+mortno+unem+dep',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WA+I(WA**2)+WE+KL6',
			'income+I(income**2)+age+I(age**2)+youngkids+oldkids',
			'income+age+youngkids+oldkids']

mod_name = ['Approve equation №1', 'Approve equation №2', 'Approve equation №3', 
			'labour force equation №1', 'labour force equation №2', 'labour force equation №3', 
			'swiss labour force equation №1', 'swiss labour force equation №2']

mod_types = ['logit', 'probit']

digits = 3

for i, df_name in enumerate(df_names):
	for mod_type in mod_types:
		if mod_type == 'logit':
			mod = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
		else:
			mod = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])

		res = mod.fit(disp=False)

		display(Markdown(f'### {mod_name[i]} ({mod_type})'))
		display(Markdown(f'Для датасет `{df_name}` оцените __{mod_type}__ регрессию `{endog_var[i]}` на `{exog_vars[i]}`. Вычислите следующие показатели качества подгонки модели:\n\n' 
		f'- R<sup>2</sup><sub>pseudo</sub>\n'
		f'- adjusted R<sup>2</sup><sub>pseudo</sub>\n'
		f'- Cox & Snell\n'
		f'- Nagelkerke/Cragg & Uhler\n'
		f'- Efron\n'
		f'- McKelvey & Zavoina\n\n'
		f'__Ответ округлите до {digits}-х десятичных знаков__\n'))

		goodness_of_fit = pd.DataFrame({'McFadden':[res.prsquared], 
										'Adj McFadden':[1-(res.llf-res.df_model-1)/res.llnull],
										'Cox-Snell':[1-np.exp(-res.llr/res.nobs)],
										'Nagelkerke-Cragg-Uhler':[(1-np.exp(-res.llr/res.nobs))/(1-np.exp(2*res.llnull/res.nobs))],
										'Efron':[1-(np.sum(res.resid_response**2))/(res.nobs*np.var(mod.endog))],
										'McKelvey-Zavoina':[ np.var(res.fittedvalues)/(np.var(res.fittedvalues)+np.pi**2/3) if mod_type=='logit' else np.var(res.fittedvalues)/(np.var(res.fittedvalues)+1) ]})
		
		display(Markdown(f'''Ответ:\n'''))
		display(Markdown(goodness_of_fit.round(digits).to_markdown(index=False)))

## Сравнение моделей

In [ ]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =[ ['appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
				'mortno+unem+dep+male+married+yjob+self',
				'appinc+I(appinc**2)+mortno+unem+dep+married',
				'appinc+I(appinc**2)+mortno+unem+dep'],
			['WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
				'WE+KL6+K618+CIT+UN+np.log(FAMINC)',
				'WA+I(WA**2)+WE+KL6+np.log(FAMINC)',
				'WA+I(WA**2)+WE+KL6'],
			['income+I(income**2)+age+I(age**2)+youngkids+oldkids+education',
				'income+age+youngkids+oldkids+education',
				'np.log(income)+age+I(age**2)+youngkids+oldkids+education',
				'np.log(income)+age+youngkids+oldkids+education']]

mod_name = ['Approve equation', 'labour force equation', 'swiss labour force equation']

mod_type = ['logit', 'probit', 'logit']

digits = 3

for i, df_name in enumerate(df_names):

	exercise_text = f'Для датасет `{df_name}` оцените __{mod_type[i]}__ регрессии `{endog_var[i]}` на\n'

	aic = []
	bic = []
	adj_pr2 = []

	for exog_var in exog_vars[i]:
		exercise_text += f'- `'+exog_var+f'`\n'

		if mod_type[i] == 'logit':
			mod = logit(formula=endog_var[i]+'~'+exog_var, data=df_dict[df_name])
		else:
			mod = probit(formula=endog_var[i]+'~'+exog_var, data=df_dict[df_name])

		res = mod.fit(disp=False)

		aic.append(res.aic)
		bic.append(res.bic)
		adj_pr2.append( 1-(res.llf-res.df_model-1)/res.llnull )

	exercise_text += f'''\n Для каждой модели вычислите показатели информационных критериев AIC и BIC и adjusted R<sup>2</sup><sub>pseudo</sub>. 
	__Ответ округлите до {digits}-х десятичных знаков__. Какая модель предпочтительней по информационных критериям и adjusted R<sup>2</sup><sub>pseudo</sub> '''

	display(Markdown(f'### {mod_name[i]} ({mod_type[i]})'))
	display(Markdown(exercise_text))

	aic, bic, adj_pr2 = np.array(aic), np.array(bic), np.array(adj_pr2)

	values_ic_r2 = pd.DataFrame({'Model #':np.arange(len(exog_vars[i])), 'AIC':aic, 'BIC':bic, 'Adj McFadden':adj_pr2})
	best_mod_ind = pd.DataFrame({'AIC': np.where(aic == aic.min()), 'BIC': np.where(bic == bic.min()), 'Adj McFadden': np.where(adj_pr2 == adj_pr2.max())})

		
	display(Markdown(f'''Ответ:\n'''))
	display(Markdown(values_ic_r2.round(digits).to_markdown(index=False)))
	display(Markdown(f'''Предпочтительная модель:\n'''))
	display(Markdown(best_mod_ind.round(digits).to_markdown(index=False)))
